# Notebook 7 — Training SSD MobileNet V2 FPNLite 320×320 dari Output IS-Net (`1b_pcv_klasik_isnet.ipynb`) + Dataset BBox Berlabel

Notebook Google Colab ini menghubungkan hasil segmentasi **IS-Net (`1b_pcv_klasik_isnet.ipynb`)** ke tahap pelatihan **SSD MobileNet V2 FPNLite 320×320**, sehingga menghasilkan:
1. **Dataset BBox IS-Net yang Sudah Berlabel Kelas (`Mobil`, `Truk`, `Bus`)** — lengkap dalam format `CSV`, `JSON`, `COCO JSON`, dan gambar preview ber-BBox + label kelas di `MyDrive/ferqueue/pcv-ssd-isnet/dataset_berlabel/`.
2. **Model SSD MobileNet V2 Hasil Latih Output IS-Net (`pcv-model-isnet-<ckpt>.zip`)** — 100% kompatibel dengan `pcv.py import-model` sehingga bisa langsung dipasang ke proyek **FERQUEUE** dan dibandingkan secara adil melawan model sebelumnya (`v4`) di **Notebook 8 (`8_perbandingan_ssd_isnet_vs_sebelumnya.ipynb`)**.

### Dua Mode Pipeline Training (`TRAINING_PIPELINE_MODE`)
- **`"isnet_bbox_rgb"` *(★ Default — Direkomendasikan untuk deploy langsung ke FERQUEUE)***:
  - Gambar latih tetap **RGB asli**, sedangkan **bounding box latih diambil dari output IS-Net (NB 1B)** dan dipasangkan otomatis dengan **label kelas (`Mobil`, `Truk`, `Bus`)**.
  - Keunggulan: Model hasil training bisa langsung dipakai kamera live FERQUEUE (`live_api.py` / `pcv.py predict`) secara **real-time** tanpa beban latency IS-Net tambahan di setiap frame.
- **`"isnet_masked_rgb"` *(Opsional Eksperimen — Background di-Masking IS-Net)***:
  - Selain memakai BBox berlabel dari IS-Net, background gambar latih juga dihapus/dinetralkan `(114, 114, 114)` oleh model ONNX IS-Net (`isnet-general-use`) sebelum dimasukkan ke `train_isnet.tfrecord`.
  - Cocok jika kamu ingin menguji skenario proyek di mana kamera juga menjalankan IS-Net terlebih dahulu sebelum SSD MobileNet V2 (otomatis terdeteksi dan dibandingkan di Notebook 8!).

### Persiapan Sebelum Menjalankan
1. Aktifkan GPU Colab: **Runtime → Change runtime type → T4 GPU** (atau L4 / A100).
2. Pastikan output Notebook 1B tersedia di Google Drive:
   ```text
   MyDrive/ferqueue/output_pcv_isnet/
   └── 07_hasil_bounding_box_semua.json   (atau 07_ringkasan_segmentasi_isnet.csv / nb1b_pcv_isnet_outputs.zip)
   ```
3. Pastikan dataset `v4` tersedia di Google Drive (sama seperti `train_colab_v4.ipynb`):
   ```text
   MyDrive/ferqueue/pcv-coco-colab-v4/
   ├── label_map.pbtxt
   └── records/
       ├── train.tfrecord
       └── test.tfrecord
   ```
   *(Jika tidak ada `train.tfrecord` v4, tetapi CSV/JSON IS-Net sudah punya kolom `Label` dan gambar ada di `MyDrive/ferqueue/gambar/`, notebook otomatis berjalan dalam **Mode Mandiri** dengan split 70/30 terstratifikasi).*
4. Jalankan cell berurutan. **Checkpoint training disimpan ke Google Drive** (`MyDrive/ferqueue/pcv-ssd-isnet/training`), sehingga aman dilanjutkan (*resume*) jika Colab terputus.

## Cell 1 — Cek GPU, Mount Google Drive, & Konfigurasi Pipeline Training

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

# 1. Pastikan GPU aktif
gpu_check = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU tidak terdeteksi! Buka menu Runtime -> Change runtime type -> pilih T4 GPU, lalu ulangi Cell 1."
    )
print(gpu_check.stdout)

# 2. Mount Google Drive
drive.mount("/content/drive")

# 3. Pilih mode integrasi IS-Net untuk data training:
#    - "isnet_bbox_rgb"   : BBox latih dari IS-Net NB 1B + label kelas (Mobil/Truk/Bus), gambar tetap RGB asli.
#                           ★ Direkomendasikan: langsung kompatibel & real-time di kamera FERQUEUE.
#    - "isnet_masked_rgb" : BBox latih dari IS-Net + label kelas + background gambar di-masking oleh IS-Net.
TRAINING_PIPELINE_MODE = "isnet_bbox_rgb"
assert TRAINING_PIPELINE_MODE in ("isnet_bbox_rgb", "isnet_masked_rgb")

# 4. Lokasi sumber input & folder simpan di Google Drive
ISNET_SOURCE_MODE = "auto"  # Pilihan: "auto" | "drive" | "upload"
DRIVE_ISNET_DIR = Path("/content/drive/MyDrive/ferqueue/output_pcv_isnet")
DRIVE_V4_DIR = Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4")
DRIVE_IMAGES_DIR = Path("/content/drive/MyDrive/ferqueue/gambar")  # Opsional jika Mode Mandiri tanpa TFRecord v4

# Folder output utama di Google Drive (otomatis terpisah bila memilih mode masked agar tidak menimpa)
RUN_FOLDER_NAME = "pcv-ssd-isnet" if TRAINING_PIPELINE_MODE == "isnet_bbox_rgb" else "pcv-ssd-isnet-masked"
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/ferqueue") / RUN_FOLDER_NAME
MODEL_DIR = DRIVE_RUN_DIR / "training"
REPORTS_DIR = DRIVE_RUN_DIR / "reports"
AUDIT_DIR = DRIVE_RUN_DIR / "audit"
LABELED_EXPORT_DIR = DRIVE_RUN_DIR / "dataset_berlabel"

# Folder kerja lokal di SSD Colab (I/O super cepat)
WORK_DIR = Path("/content/ferqueue_isnet_train")
VENV_DIR = Path("/content/.venv-pcv")
RECORDS_DIR = WORK_DIR / "records"
ISNET_WORK_DIR = WORK_DIR / "isnet_input"

for d in (
    WORK_DIR, RECORDS_DIR, ISNET_WORK_DIR,
    DRIVE_RUN_DIR, MODEL_DIR, REPORTS_DIR, AUDIT_DIR, LABELED_EXPORT_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

# 5. Kebijakan pembangunan dataset dari output IS-Net
# - "skip"        : HANYA melatih gambar yang punya BBox valid dari IS-Net (+ background negatif).
#                   Murni memakai BBox IS-Net tanpa campur tangan BBox manual GT. (★ Direkomendasikan)
# - "fallback_gt" : Jika ada gambar train yang belum ada di output IS-Net, pakai BBox GT manual.
UNMATCHED_POLICY = "skip"
KEEP_BACKGROUND_NEGATIVE = True    # Pertahankan gambar non-kendaraan (0 box) untuk menekan False Positive
MIN_ISNET_BBOX_AREA_RATIO = 0.005  # Buang noise box < 0.5% luas gambar
MASK_BG_COLOR = (114, 114, 114)    # Warna latar netral bila TRAINING_PIPELINE_MODE == "isnet_masked_rgb"
SEED = 42

# 6. Hyperparameter training (IDENTIK dengan train_colab_v4.ipynb agar perbandingan di Notebook 8 adil)
CLASSES = ["Mobil", "Truk", "Bus"]
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
PREPROCESSING = {
    "denoise_kernel": 0,
    "clahe_clip_limit": 0,
    "clahe_grid_size": 8,
    "gamma": 1.0,
}

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 0.004
FREEZE_BATCHNORM = False  # WAJIB False agar BatchNorm box_predictor baru ter-update (mencegah skor kolaps)
CHECKPOINT_EVERY_N = 100
HORIZONTAL_FLIP = True
BRIGHTNESS_DELTA = 0.1

print("Konfigurasi siap!")
print(f"  TRAINING_PIPELINE_MODE : {TRAINING_PIPELINE_MODE}")
print(f"  DRIVE_ISNET_DIR        : {DRIVE_ISNET_DIR}")
print(f"  DRIVE_V4_DIR           : {DRIVE_V4_DIR}")
print(f"  DRIVE_RUN_DIR          : {DRIVE_RUN_DIR}")
print(f"  LABELED_EXPORT_DIR     : {LABELED_EXPORT_DIR}")
print(f"  UNMATCHED_POLICY       : {UNMATCHED_POLICY} | KEEP_BACKGROUND_NEGATIVE={KEEP_BACKGROUND_NEGATIVE}")

## Cell 2 — Muat & Verifikasi Output IS-Net (`1b_pcv_klasik_isnet.ipynb`)

Mendukung seluruh format ekspor dari Notebook 1B secara otomatis:
1. `07_hasil_bounding_box_semua.json` (JSON gabungan)
2. `07_ringkasan_segmentasi_isnet.csv` (CSV rekap)
3. `nb1b_pcv_isnet_outputs.zip` (arsip ZIP lengkap termasuk `per_gambar/*.json`)

In [ ]:
import json
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import files as colab_files


def _clamp01(v: float) -> float:
    return max(0.0, min(1.0, float(v)))


def _valid_box_norm(x1, y1, x2, y2, min_area: float = MIN_ISNET_BBOX_AREA_RATIO):
    x1, y1, x2, y2 = _clamp01(x1), _clamp01(y1), _clamp01(x2), _clamp01(y2)
    bw, bh = x2 - x1, y2 - y1
    if bw < 0.01 or bh < 0.01 or (bw * bh) < min_area:
        return None
    return [round(x1, 6), round(y1, 6), round(x2, 6), round(y2, 6)]


def _normalize_label(raw_label) -> str | None:
    if raw_label is None:
        return None
    s = str(raw_label).strip()
    if not s or s.lower() in ("nan", "none", "null"):
        return None
    mapping = {c.lower(): c for c in CLASSES}
    return mapping.get(s.lower(), s)


def parse_isnet_json_file(json_path: Path) -> list[dict]:
    payload = json.loads(json_path.read_text(encoding="utf-8"))
    if isinstance(payload, dict) and "results" in payload:
        items = payload["results"]
    elif isinstance(payload, list):
        items = payload
    elif isinstance(payload, dict) and "file_name" in payload:
        items = [payload]
    else:
        raise ValueError(f"Format JSON IS-Net tidak dikenali: {json_path}")

    parsed = []
    for rec in items:
        fname = str(rec.get("file_name") or rec.get("File_Name") or "").strip()
        if not fname:
            continue
        img_w = int(rec.get("image_width") or rec.get("Image_W") or 0)
        img_h = int(rec.get("image_height") or rec.get("Image_H") or 0)
        raw_objs = rec.get("objects") or []
        boxes = []
        for obj in raw_objs:
            norm = obj.get("bbox_norm_xyxy")
            if norm and len(norm) == 4:
                vb = _valid_box_norm(norm[0], norm[1], norm[2], norm[3])
            else:
                px = obj.get("bbox_xyxy_px")
                if px and len(px) == 4 and img_w > 0 and img_h > 0:
                    vb = _valid_box_norm(px[0] / img_w, px[1] / img_h, px[2] / img_w, px[3] / img_h)
                else:
                    vb = None
            if vb is not None:
                boxes.append({
                    "bbox_norm_xyxy": vb,
                    "label": _normalize_label(obj.get("label")),
                    "aspect_ratio": float(obj.get("aspect_ratio") or 0.0),
                })
        parsed.append({
            "file_name": Path(fname).name,
            "image_width": img_w,
            "image_height": img_h,
            "mask_coverage_pct": float(rec.get("mask_coverage_pct") or 0.0),
            "boxes": boxes,
        })
    return parsed


def parse_isnet_csv_file(csv_path: Path) -> list[dict]:
    df = pd.read_csv(csv_path)
    parsed = []
    for _, row in df.iterrows():
        fname = str(row.get("File_Name") or row.get("file_name") or "").strip()
        if not fname or fname.lower() == "nan":
            continue
        img_w = int(row.get("Image_W") or 0)
        img_h = int(row.get("Image_H") or 0)
        boxes = []
        if all(col in df.columns for col in ("Norm_xmin", "Norm_ymin", "Norm_xmax", "Norm_ymax")):
            vb = _valid_box_norm(row["Norm_xmin"], row["Norm_ymin"], row["Norm_xmax"], row["Norm_ymax"])
        elif all(col in df.columns for col in ("BBox_xmin", "BBox_ymin", "BBox_xmax", "BBox_ymax")) and img_w > 0 and img_h > 0:
            vb = _valid_box_norm(
                float(row["BBox_xmin"]) / img_w,
                float(row["BBox_ymin"]) / img_h,
                float(row["BBox_xmax"]) / img_w,
                float(row["BBox_ymax"]) / img_h,
            )
        else:
            vb = None
        if vb is not None:
            boxes.append({
                "bbox_norm_xyxy": vb,
                "label": _normalize_label(row.get("Label")),
                "aspect_ratio": float(row.get("Aspect_Ratio") or 0.0),
            })
        parsed.append({
            "file_name": Path(fname).name,
            "image_width": img_w,
            "image_height": img_h,
            "mask_coverage_pct": float(row.get("Cakupan_%") or 0.0),
            "boxes": boxes,
        })
    return parsed


def load_isnet_records(source_mode: str):
    candidates = [
        DRIVE_ISNET_DIR / "07_hasil_bounding_box_semua.json",
        DRIVE_ISNET_DIR / "07_ringkasan_segmentasi_isnet.csv",
        DRIVE_ISNET_DIR / "nb1b_pcv_isnet_outputs.zip",
        Path("/content/outputs_presentasi/nb1b_pcv_isnet/07_hasil_bounding_box_semua.json"),
        Path("/content/outputs_presentasi/nb1b_pcv_isnet/07_ringkasan_segmentasi_isnet.csv"),
        Path("/content/drive/MyDrive/ferqueue/07_hasil_bounding_box_semua.json"),
        Path("/content/drive/MyDrive/ferqueue/07_ringkasan_segmentasi_isnet.csv"),
        Path("/content/drive/MyDrive/ferqueue/nb1b_pcv_isnet_outputs.zip"),
    ]

    chosen = None
    if source_mode in ("auto", "drive"):
        for c in candidates:
            if c.is_file():
                chosen = c
                break

    if chosen is None:
        if source_mode == "drive":
            tried = "\n  - ".join(str(c) for c in candidates[:3])
            raise FileNotFoundError(
                f"Output IS-Net tidak ditemukan di Drive. Pastikan salah satu file berikut ada:\n  - {tried}"
            )
        print(
            "Output IS-Net belum ditemukan di Google Drive.\n"
            "Silakan unggah 07_hasil_bounding_box_semua.json, 07_ringkasan_segmentasi_isnet.csv, atau nb1b_pcv_isnet_outputs.zip:"
        )
        uploaded = colab_files.upload()
        if not uploaded:
            raise RuntimeError("Tidak ada file output IS-Net yang diunggah.")
        up_name, up_bytes = next(iter(uploaded.items()))
        chosen = ISNET_WORK_DIR / Path(up_name).name
        chosen.write_bytes(up_bytes)

    print(f"Memuat output IS-Net dari: {chosen}")
    if chosen.suffix.lower() == ".zip":
        extract_dir = ISNET_WORK_DIR / "extracted_nb1b"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(chosen) as zf:
            zf.extractall(extract_dir)
        json_hits = list(extract_dir.rglob("07_hasil_bounding_box_semua.json"))
        csv_hits = list(extract_dir.rglob("07_ringkasan_segmentasi_isnet.csv"))
        per_img_jsons = sorted(extract_dir.rglob("per_gambar/*.json"))
        if json_hits:
            return parse_isnet_json_file(json_hits[0]), str(chosen)
        if csv_hits:
            return parse_isnet_csv_file(csv_hits[0]), str(chosen)
        if per_img_jsons:
            merged = []
            for pj in per_img_jsons:
                merged.extend(parse_isnet_json_file(pj))
            return merged, str(chosen)
        raise FileNotFoundError("Di dalam ZIP tidak ditemukan JSON/CSV output IS-Net.")
    elif chosen.suffix.lower() == ".json":
        return parse_isnet_json_file(chosen), str(chosen)
    elif chosen.suffix.lower() == ".csv":
        return parse_isnet_csv_file(chosen), str(chosen)
    else:
        raise ValueError(f"Ekstensi file tidak didukung: {chosen.suffix}")


isnet_records, ISNET_SOURCE_PATH = load_isnet_records(ISNET_SOURCE_MODE)
if not isnet_records:
    raise RuntimeError("Output IS-Net kosong — tidak ada record gambar yang bisa dipakai.")

ISNET_NORMALIZED_JSON = WORK_DIR / "isnet_normalized_records.json"
ISNET_NORMALIZED_JSON.write_text(
    json.dumps({"source": ISNET_SOURCE_PATH, "records": isnet_records}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

n_total = len(isnet_records)
n_with_box = sum(1 for r in isnet_records if r["boxes"])
n_labeled = sum(1 for r in isnet_records if any(b.get("label") in CLASSES for b in r["boxes"]))
mean_cov = sum(r["mask_coverage_pct"] for r in isnet_records) / max(1, n_total)

print("\n=== RINGKASAN OUTPUT IS-NET (DARI NB 1B) ===")
print(f"Sumber               : {ISNET_SOURCE_PATH}")
print(f"Total record gambar  : {n_total}")
print(f"Punya BBox valid     : {n_with_box} ({100.0 * n_with_box / n_total:.1f}%)")
print(f"Sudah berlabel awal  : {n_labeled} ({100.0 * n_labeled / n_total:.1f}%) -> label kelas akan dipasangkan di Cell 5")
print(f"Rata-rata cakupan    : {mean_cov:.1f}% luas gambar")

## Cell 3 — Pasang Environment Python 3.11 + TensorFlow 2.15.1 (CUDA) Terisolasi

Sama persis dengan `train_colab_v4.ipynb` agar environment training 100% kompatibel dengan TensorFlow Object Detection API (TF 2.15.1 + Keras 2.15 + Protobuf 3.20.3) tanpa bentrok dengan runtime bawaan Colab dan tanpa perlu restart runtime.

In [ ]:
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/ferqueue_isnet_train")
VENV_DIR = Path("/content/.venv-pcv")
VENV_PY = VENV_DIR / "bin/python"

# 1. Pasang uv (manajer Python & pip super cepat)
subprocess.run(["pip", "install", "-q", "uv"], check=True)

# 2. Buat venv Python 3.11 jika belum ada
if not VENV_PY.is_file():
    print("Mengunduh Python 3.11 dan membuat venv /content/.venv-pcv ...")
    subprocess.run(["uv", "python", "install", "3.11"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.11", "--seed", str(VENV_DIR)], check=True)
else:
    print("Venv sudah tersedia:", VENV_PY)

# 3. Tulis requirements yang identik dengan FERQUEUE (training-cuda.txt)
extra_reqs = "\nrembg[gpu]\n" if TRAINING_PIPELINE_MODE == "isnet_masked_rgb" else ""
req_file = WORK_DIR / "requirements-colab.txt"
req_file.write_text(
    """Pillow==10.4.0
numpy==1.26.4
opencv-python-headless==4.10.0.84
setuptools==80.9.0
tensorboard==2.15.2
tensorflow[and-cuda]==2.15.1
tf-models-official==2.15.0
tensorflow-text==2.15.0
tensorflow-io==0.36.0
tensorflow-datasets==4.9.3
tensorflow-metadata==1.14.0
keras==2.15.0
tf-keras==2.15.1
protobuf==3.20.3
tf-slim==1.1.0
pycocotools==2.0.7
lvis==0.5.3
lxml==5.3.0
contextlib2==21.6.0
pyparsing==2.4.7
gin-config==0.5.0
PyYAML==6.0.2
six==1.16.0
scipy==1.11.4
matplotlib==3.8.4
pandas==2.1.4
""" + extra_reqs,
    encoding="utf-8",
)

print("Memasang TensorFlow 2.15.1 (CUDA) & dependensi Object Detection API (tunggu ~1-2 menit)...")
subprocess.run(
    ["uv", "pip", "install", "--python", str(VENV_PY), "-r", str(req_file)],
    check=True,
)
print("Selesai memasang environment Python 3.11 di", VENV_DIR)

## Cell 4 — Unduh & Compile TensorFlow Object Detection API + Patch `tf-slim` + Verifikasi GPU

In [ ]:
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/ferqueue_isnet_train")
VENV_DIR = Path("/content/.venv-pcv")
VENV_PY = VENV_DIR / "bin/python"

TF_MODELS_COMMIT = "40695ad8964145a90ae1abbcc740c086e7364eac"
TF_MODELS_URL = f"https://github.com/tensorflow/models/archive/{TF_MODELS_COMMIT}.zip"
TF_MODELS_DIR = WORK_DIR / "tensorflow-models"
PROTOC_URL = "https://github.com/protocolbuffers/protobuf/releases/download/v3.20.3/protoc-3.20.3-linux-x86_64.zip"
PROTOC_DIR = WORK_DIR / "protoc-3.20.3"

# 1. Unduh sumber TensorFlow Models (commit resmi proyek FERQUEUE)
if not (TF_MODELS_DIR / "research/object_detection/protos/pipeline.proto").is_file():
    zip_path = WORK_DIR / "tf-models.zip"
    print("Mengunduh TensorFlow Models API...")
    urllib.request.urlretrieve(TF_MODELS_URL, zip_path)
    extract_tmp = WORK_DIR / "tf-models-extract"
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_tmp)
    extracted_root = extract_tmp / f"models-{TF_MODELS_COMMIT}"
    if TF_MODELS_DIR.exists():
        shutil.rmtree(TF_MODELS_DIR)
    shutil.move(str(extracted_root), str(TF_MODELS_DIR))
    shutil.rmtree(extract_tmp, ignore_errors=True)
    zip_path.unlink(missing_ok=True)
    print("Sumber Object Detection API siap:", TF_MODELS_DIR)
else:
    print("Sumber Object Detection API sudah tersedia:", TF_MODELS_DIR)

# 2. Unduh protoc 3.20.3 dan compile *.proto
protoc_bin = PROTOC_DIR / "bin/protoc"
if not protoc_bin.is_file():
    pzip = WORK_DIR / "protoc.zip"
    print("Mengunduh protoc 3.20.3...")
    urllib.request.urlretrieve(PROTOC_URL, pzip)
    PROTOC_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(pzip) as zf:
        zf.extractall(PROTOC_DIR)
    protoc_bin.chmod(0o755)
    pzip.unlink(missing_ok=True)

research_dir = TF_MODELS_DIR / "research"
if not (research_dir / "object_detection/protos/pipeline_pb2.py").is_file():
    print("Mengompilasi protobuf Object Detection API...")
    protos = sorted(str(p) for p in (research_dir / "object_detection/protos").glob("*.proto"))
    subprocess.run(
        [str(protoc_bin), f"--proto_path={research_dir}", f"--python_out={research_dir}", *protos],
        check=True,
    )
    print("Kompilasi protobuf selesai.")
else:
    print("Protobuf sudah terkompilasi.")

# 3. Helper untuk menjalankan skrip di dalam .venv-pcv dengan CUDA & OD API
def build_venv_env():
    env = os.environ.copy()
    site_packages = list((VENV_DIR / "lib").glob("python3.*/site-packages"))
    nvidia_libs = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/lib"))]
    nvidia_bins = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/bin"))]

    python_paths = [str(research_dir), str(research_dir / "slim")]
    if env.get("PYTHONPATH"):
        python_paths.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = ":".join(python_paths)

    ld_paths = nvidia_libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
    env["LD_LIBRARY_PATH"] = ":".join(ld_paths)

    bin_paths = [str(VENV_DIR / "bin")] + nvidia_bins + ([env["PATH"]] if env.get("PATH") else [])
    env["PATH"] = ":".join(bin_paths)
    env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    env["TF_CPP_MIN_LOG_LEVEL"] = "1"
    return env


def run_venv(args, stream=True):
    env = build_venv_env()
    cmd = [str(VENV_PY), *args]
    if not stream:
        return subprocess.run(cmd, env=env, check=True, text=True, capture_output=True)
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


# 4. Patch tf-slim decoder + verifikasi GPU & JPEG/PNG decoder
verify_script = WORK_DIR / "verify_env.py"
verify_script.write_text(
    r"""import importlib.util
from pathlib import Path

spec = importlib.util.find_spec("tf_slim")
decoder_path = Path(spec.origin).parent / "data/tfexample_decoder.py"
src = decoder_path.read_text(encoding="utf-8")
old_imp = "from tensorflow.python.ops import control_flow_ops"
new_imp = "from tensorflow.compat import v1 as control_flow_ops"
if new_imp not in src and old_imp in src:
    decoder_path.write_text(src.replace(old_imp, new_imp), encoding="utf-8")
    print("Patch tf-slim diterapkan.")
else:
    print("Patch tf-slim sudah sesuai.")

import tensorflow as tf
from tf_slim.data import tfexample_decoder

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "| GPU terdeteksi:", gpus)
if not gpus:
    raise RuntimeError("TensorFlow tidak mendeteksi GPU! Periksa runtime GPU Colab.")

handler = tfexample_decoder.Image(channels=3)
dummy = tf.zeros([4, 6, 3], dtype=tf.uint8)

@tf.function
def test_decode(encoded, fmt):
    return handler.tensors_to_item({"image/encoded": encoded, "image/format": fmt})

for fmt, enc in ((b"jpeg", tf.io.encode_jpeg(dummy)), (b"png", tf.io.encode_png(dummy))):
    out = test_decode(enc, tf.constant(fmt)).numpy()
    assert out.shape == (4, 6, 3), f"Decoder shape salah: {out.shape}"
print("Decoder tf-slim JPEG/PNG OK.")
""",
    encoding="utf-8",
)

run_venv([str(verify_script)])

## Cell 5 — Pasangkan BBox IS-Net dengan Label Kelas (`Mobil`, `Truk`, `Bus`), Ekspor Dataset Berlabel, & Bangun TFRecord

Cell ini melakukan 4 hal sekaligus:
1. **Memberi label kelas (`Mobil`, `Truk`, `Bus`) pada BBox output IS-Net** dengan mencocokkan `file_name` terhadap `train.tfrecord` v4 (atau memakai kolom `Label` jika menjalankan Mode Mandiri).
2. **Mengekspor dataset BBox IS-Net yang sudah berlabel** ke `MyDrive/ferqueue/pcv-ssd-isnet/dataset_berlabel/`:
   - `isnet_dataset_berlabel.csv` (tabel lengkap berisi `Label`, koordinat BBox IS-Net piksel & ternormalisasi, serta IoU terhadap GT manual)
   - `isnet_dataset_berlabel.json` (format identik NB 1B namun `label` sudah terisi `Mobil`/`Truk`/`Bus`!)
   - `instances_train_isnet_coco.json` (format anotasi **COCO JSON** siap pakai ulang)
3. **Membangun `train_isnet.tfrecord` & menyalin `test.tfrecord` Ground Truth bersih** agar evaluasi model IS-Net benar-benar setara dengan model `v4` sebelumnya. (Jika `TRAINING_PIPELINE_MODE == "isnet_masked_rgb"`, background gambar latih juga di-masking otomatis oleh IS-Net).
4. **Menampilkan 4 sampel visual** (BBox IS-Net berlabel kelas + tag warna per kelas vs referensi GT manual) dan statistik IoU BBox IS-Net vs GT!

In [ ]:
import json
import math
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

LOCAL_TRAIN_TFREC = RECORDS_DIR / "train_isnet.tfrecord"
LOCAL_TEST_TFREC = RECORDS_DIR / "test.tfrecord"
LOCAL_LABEL_MAP = WORK_DIR / "label_map.pbtxt"
PREVIEW_DIR = WORK_DIR / "preview_samples"
AUDIT_JSON_PATH = AUDIT_DIR / "isnet_dataset_audit.json"
AUDIT_CSV_PATH = AUDIT_DIR / "isnet_bbox_alignment.csv"

LABELED_CSV_PATH = LABELED_EXPORT_DIR / "isnet_dataset_berlabel.csv"
LABELED_JSON_PATH = LABELED_EXPORT_DIR / "isnet_dataset_berlabel.json"
LABELED_COCO_PATH = LABELED_EXPORT_DIR / "instances_train_isnet_coco.json"


def find_optional(candidates):
    for p in candidates:
        if p.is_file():
            return p
    return None


v4_train_src = find_optional([DRIVE_V4_DIR / "records/train.tfrecord", DRIVE_V4_DIR / "train.tfrecord"])
v4_test_src = find_optional([DRIVE_V4_DIR / "records/test.tfrecord", DRIVE_V4_DIR / "test.tfrecord"])
v4_label_map_src = find_optional([DRIVE_V4_DIR / "label_map.pbtxt"])

local_v4_train = None
local_v4_test = None
if v4_train_src and v4_test_src:
    local_v4_train = WORK_DIR / "v4_train_src.tfrecord"
    local_v4_test = WORK_DIR / "v4_test_src.tfrecord"
    for src, dst in [(v4_train_src, local_v4_train), (v4_test_src, local_v4_test)]:
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            print(f"Menyalin {src.name} ({src.stat().st_size / (1024**2):.1f} MB) ke SSD lokal Colab...")
            shutil.copy2(src, dst)
        else:
            print(f"Sudah ada di SSD lokal: {dst.name}")

if v4_label_map_src:
    shutil.copy2(v4_label_map_src, LOCAL_LABEL_MAP)
else:
    pbtxt = "\n".join(
        f"item {{\n  id: {idx}\n  name: '{cls_name}'\n}}"
        for idx, cls_name in enumerate(CLASSES, start=1)
    ) + "\n"
    LOCAL_LABEL_MAP.write_text(pbtxt, encoding="utf-8")

builder_params_path = WORK_DIR / "build_tfrecord_params.json"
builder_params_path.write_text(
    json.dumps(
        {
            "classes": CLASSES,
            "pipeline_mode": TRAINING_PIPELINE_MODE,
            "mask_bg_color": list(MASK_BG_COLOR),
            "isnet_json": str(ISNET_NORMALIZED_JSON),
            "v4_train_tfrecord": str(local_v4_train) if local_v4_train else "",
            "v4_test_tfrecord": str(local_v4_test) if local_v4_test else "",
            "images_dir": str(DRIVE_IMAGES_DIR),
            "out_train_tfrecord": str(LOCAL_TRAIN_TFREC),
            "out_test_tfrecord": str(LOCAL_TEST_TFREC),
            "unmatched_policy": UNMATCHED_POLICY,
            "keep_background_negative": KEEP_BACKGROUND_NEGATIVE,
            "seed": SEED,
            "audit_json": str(AUDIT_JSON_PATH),
            "audit_csv": str(AUDIT_CSV_PATH),
            "labeled_csv": str(LABELED_CSV_PATH),
            "labeled_json": str(LABELED_JSON_PATH),
            "labeled_coco_json": str(LABELED_COCO_PATH),
            "preview_dir": str(PREVIEW_DIR),
        },
        indent=2,
    ),
    encoding="utf-8",
)

builder_script = WORK_DIR / "build_isnet_tfrecord.py"
builder_script.write_text(
    r"""import csv
import hashlib
import io
import json
import random
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

params = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CLASSES = params["classes"]
PIPELINE_MODE = params["pipeline_mode"]
MASK_BG_COLOR = tuple(int(c) for c in params["mask_bg_color"])
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES, start=1)}
ID_TO_CLASS = {i: c for i, c in enumerate(CLASSES, start=1)}
UNMATCHED_POLICY = params["unmatched_policy"]
KEEP_BG = bool(params["keep_background_negative"])
SEED = int(params["seed"])

# Siapkan sesi IS-Net hanya bila TRAINING_PIPELINE_MODE == "isnet_masked_rgb"
isnet_session = None
if PIPELINE_MODE == "isnet_masked_rgb":
    import onnxruntime as ort
    from rembg import new_session, remove

    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if "CUDAExecutionProvider" in ort.get_available_providers()
        else ["CPUExecutionProvider"]
    )
    print(f"Mode isnet_masked_rgb aktif -> memuat IS-Net ({providers[0]}) untuk masking background gambar latih...")
    isnet_session = new_session("isnet-general-use", providers=providers)


def apply_isnet_mask_to_bytes(encoded_bytes: bytes) -> tuple[bytes, bytes]:
    # Hapus background dengan IS-Net dan ganti dengan warna netral MASK_BG_COLOR.
    with Image.open(io.BytesIO(encoded_bytes)) as im:
        rgb = im.convert("RGB")
    rgba = remove(rgb, session=isnet_session)
    arr = np.asarray(rgba)
    alpha = arr[:, :, 3:4].astype(np.float32) / 255.0
    fg = arr[:, :, :3].astype(np.float32)
    bg = np.full_like(fg, MASK_BG_COLOR, dtype=np.float32)
    comp = np.clip(fg * alpha + bg * (1.0 - alpha), 0, 255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(comp).save(buf, format="PNG")
    return buf.getvalue(), b"png"


isnet_payload = json.loads(Path(params["isnet_json"]).read_text(encoding="utf-8"))
isnet_records = isnet_payload["records"]

by_exact = {}
by_lower = {}
stem_buckets = defaultdict(list)
for rec in isnet_records:
    fn = Path(rec["file_name"]).name
    by_exact[fn] = rec
    by_lower[fn.lower()] = rec
    stem_buckets[Path(fn).stem.lower()].append(rec)
by_unique_stem = {k: v[0] for k, v in stem_buckets.items() if len(v) == 1}


def find_isnet_record(file_path_str: str):
    name = Path(file_path_str.replace("\\", "/")).name
    if name in by_exact:
        return by_exact[name], "exact"
    if name.lower() in by_lower:
        return by_lower[name.lower()], "lower"
    stem = Path(name).stem.lower()
    if stem in by_unique_stem:
        return by_unique_stem[stem], "stem"
    return None, "missing"


def compute_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 1e-9 else 0.0


def _bytes(vals):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=vals))


def _ints(vals):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=vals))


def _floats(vals):
    return tf.train.Feature(float_list=tf.train.FloatList(value=vals))


def make_tf_example(height, width, filename_str, source_id_str, encoded_bytes, fmt_bytes, boxes):
    return tf.train.Example(
        features=tf.train.Features(
            feature={
                "image/height": _ints([int(height)]),
                "image/width": _ints([int(width)]),
                "image/filename": _bytes([filename_str.encode("utf-8")]),
                "image/source_id": _bytes([source_id_str.encode("utf-8")]),
                "image/encoded": _bytes([encoded_bytes]),
                "image/format": _bytes([fmt_bytes]),
                "image/object/bbox/xmin": _floats([float(b["xmin"]) for b in boxes]),
                "image/object/bbox/xmax": _floats([float(b["xmax"]) for b in boxes]),
                "image/object/bbox/ymin": _floats([float(b["ymin"]) for b in boxes]),
                "image/object/bbox/ymax": _floats([float(b["ymax"]) for b in boxes]),
                "image/object/class/text": _bytes([ID_TO_CLASS[int(b["class_id"])].encode("utf-8") for b in boxes]),
                "image/object/class/label": _ints([int(b["class_id"]) for b in boxes]),
                "image/object/is_crowd": _ints([0] * len(boxes)),
            }
        )
    )


feature_spec = {
    "image/height": tf.io.FixedLenFeature([], tf.int64),
    "image/width": tf.io.FixedLenFeature([], tf.int64),
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/source_id": tf.io.FixedLenFeature([], tf.string, default_value=b""),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/format": tf.io.FixedLenFeature([], tf.string, default_value=b"png"),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

preview_dir = Path(params["preview_dir"])
if preview_dir.exists():
    shutil.rmtree(preview_dir)
preview_dir.mkdir(parents=True, exist_ok=True)
preview_meta = []

v4_train = params["v4_train_tfrecord"]
v4_test = params["v4_test_tfrecord"]
out_train = Path(params["out_train_tfrecord"])
out_test = Path(params["out_test_tfrecord"])
out_train.parent.mkdir(parents=True, exist_ok=True)

alignment_rows = []
labeled_csv_rows = []
labeled_json_results = []
coco_train_doc = {
    "info": {"description": f"FERQUEUE Train Dataset dari BBox IS-Net Berlabel ({PIPELINE_MODE})"},
    "categories": [{"id": i, "name": c, "supercategory": "vehicle"} for i, c in enumerate(CLASSES, start=1)],
    "images": [],
    "annotations": [],
}
class_counts_train = Counter()
class_counts_test = Counter()


def record_labeled_train_entry(split_name: str, file_name: str, w: int, h: int, boxes: list[dict], bbox_source: str, iou_gt):
    img_id = len(coco_train_doc["images"]) + 1
    coco_train_doc["images"].append({"id": img_id, "file_name": file_name, "width": int(w), "height": int(h)})
    objs_json = []
    for obj_i, b in enumerate(boxes, start=1):
        cname = ID_TO_CLASS[int(b["class_id"])]
        x1_px = round(b["xmin"] * w, 2)
        y1_px = round(b["ymin"] * h, 2)
        x2_px = round(b["xmax"] * w, 2)
        y2_px = round(b["ymax"] * h, 2)
        bw_px = max(1.0, round(x2_px - x1_px, 2))
        bh_px = max(1.0, round(y2_px - y1_px, 2))
        coco_train_doc["annotations"].append({
            "id": len(coco_train_doc["annotations"]) + 1,
            "image_id": img_id,
            "category_id": int(b["class_id"]),
            "bbox": [x1_px, y1_px, bw_px, bh_px],
            "area": round(bw_px * bh_px, 2),
            "iscrowd": 0,
        })
        objs_json.append({
            "object_id": obj_i,
            "label": cname,
            "class_id": int(b["class_id"]),
            "bbox_source": bbox_source,
            "bbox_xywh_px": [int(round(x1_px)), int(round(y1_px)), int(round(bw_px)), int(round(bh_px))],
            "bbox_xyxy_px": [int(round(x1_px)), int(round(y1_px)), int(round(x2_px)), int(round(y2_px))],
            "bbox_norm_xyxy": [round(b["xmin"], 6), round(b["ymin"], 6), round(b["xmax"], 6), round(b["ymax"], 6)],
        })

    primary = boxes[0] if boxes else None
    labeled_csv_rows.append({
        "No": img_id,
        "Split": split_name,
        "File_Name": file_name,
        "Image_W": int(w),
        "Image_H": int(h),
        "Label": ID_TO_CLASS[int(primary["class_id"])] if primary else "Background",
        "Class_ID": int(primary["class_id"]) if primary else 0,
        "BBox_Source": bbox_source,
        "Jumlah_Objek": len(boxes),
        "BBox_xmin": int(round(primary["xmin"] * w)) if primary else 0,
        "BBox_ymin": int(round(primary["ymin"] * h)) if primary else 0,
        "BBox_xmax": int(round(primary["xmax"] * w)) if primary else 0,
        "BBox_ymax": int(round(primary["ymax"] * h)) if primary else 0,
        "Norm_xmin": round(primary["xmin"], 6) if primary else 0.0,
        "Norm_ymin": round(primary["ymin"], 6) if primary else 0.0,
        "Norm_xmax": round(primary["xmax"], 6) if primary else 0.0,
        "Norm_ymax": round(primary["ymax"], 6) if primary else 0.0,
        "IoU_ISNet_vs_GT": round(float(iou_gt), 5) if iou_gt is not None else "",
    })
    labeled_json_results.append({
        "index": img_id,
        "split": split_name,
        "file_name": file_name,
        "image_width": int(w),
        "image_height": int(h),
        "bbox_source": bbox_source,
        "primary_label": ID_TO_CLASS[int(primary["class_id"])] if primary else "Background",
        "object_count": len(objs_json),
        "objects": objs_json,
    })


if v4_train and v4_test and Path(v4_train).is_file() and Path(v4_test).is_file():
    mode_used = f"v4_tfrecord_plus_{PIPELINE_MODE}"
    print(f"Mode pembangunan dataset: {mode_used}")

    stats = {
        "train_source_total": 0,
        "train_matched_isnet": 0,
        "train_background_kept": 0,
        "train_unmatched_skipped": 0,
        "train_unmatched_fallback_gt": 0,
        "train_written_total": 0,
        "test_total": 0,
        "test_background": 0,
        "test_matched_isnet_for_audit": 0,
    }

    tmp_train = out_train.with_suffix(".tfrecord.partial")
    with tf.io.TFRecordWriter(str(tmp_train)) as writer:
        for raw in tf.data.TFRecordDataset(v4_train):
            stats["train_source_total"] += 1
            ex = tf.io.parse_single_example(raw, feature_spec)
            h = int(ex["image/height"].numpy())
            w = int(ex["image/width"].numpy())
            fname = ex["image/filename"].numpy().decode("utf-8", errors="replace")
            base_name = Path(fname.replace("\\", "/")).name
            src_id = ex["image/source_id"].numpy().decode("utf-8", errors="replace") or hashlib.sha256(fname.encode()).hexdigest()
            encoded = ex["image/encoded"].numpy()
            fmt = ex["image/format"].numpy() or b"png"

            xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
            ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
            xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
            ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
            labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()

            gt_boxes = [
                {
                    "class_id": int(cid),
                    "xmin": float(x1),
                    "ymin": float(y1),
                    "xmax": float(x2),
                    "ymax": float(y2),
                }
                for x1, y1, x2, y2, cid in zip(xmins, ymins, xmaxs, ymaxs, labels)
            ]

            if not gt_boxes:
                if KEEP_BG:
                    enc_write, fmt_write = apply_isnet_mask_to_bytes(encoded) if PIPELINE_MODE == "isnet_masked_rgb" else (encoded, fmt)
                    tf_ex = make_tf_example(h, w, fname, src_id, enc_write, fmt_write, [])
                    writer.write(tf_ex.SerializeToString())
                    record_labeled_train_entry("train", base_name, w, h, [], "background_negative", None)
                    stats["train_background_kept"] += 1
                    stats["train_written_total"] += 1
                continue

            isnet_rec, match_type = find_isnet_record(fname)
            isnet_boxes_raw = isnet_rec["boxes"] if isnet_rec else []

            if isnet_boxes_raw:
                new_boxes = []
                primary_iou = None
                if len(gt_boxes) == 1:
                    g = gt_boxes[0]
                    best_ib = max(
                        isnet_boxes_raw,
                        key=lambda ib: compute_iou(ib["bbox_norm_xyxy"], [g["xmin"], g["ymin"], g["xmax"], g["ymax"]]),
                    )
                    ix1, iy1, ix2, iy2 = best_ib["bbox_norm_xyxy"]
                    iou_val = compute_iou([ix1, iy1, ix2, iy2], [g["xmin"], g["ymin"], g["xmax"], g["ymax"]])
                    primary_iou = iou_val
                    new_boxes.append({
                        "class_id": g["class_id"],
                        "xmin": ix1,
                        "ymin": iy1,
                        "xmax": ix2,
                        "ymax": iy2,
                    })
                    alignment_rows.append({
                        "split": "train",
                        "file_name": base_name,
                        "match_type": match_type,
                        "class": ID_TO_CLASS[g["class_id"]],
                        "iou_isnet_vs_gt": round(iou_val, 5),
                        "gt_xmin": round(g["xmin"], 5),
                        "gt_ymin": round(g["ymin"], 5),
                        "gt_xmax": round(g["xmax"], 5),
                        "gt_ymax": round(g["ymax"], 5),
                        "isnet_xmin": round(ix1, 5),
                        "isnet_ymin": round(iy1, 5),
                        "isnet_xmax": round(ix2, 5),
                        "isnet_ymax": round(iy2, 5),
                        "isnet_area_ratio": round((ix2 - ix1) * (iy2 - iy1), 5),
                        "gt_area_ratio": round((g["xmax"] - g["xmin"]) * (g["ymax"] - g["ymin"]), 5),
                    })
                else:
                    for ib in isnet_boxes_raw:
                        ix1, iy1, ix2, iy2 = ib["bbox_norm_xyxy"]
                        best_g = max(
                            gt_boxes,
                            key=lambda g: compute_iou([ix1, iy1, ix2, iy2], [g["xmin"], g["ymin"], g["xmax"], g["ymax"]]),
                        )
                        iou_val = compute_iou([ix1, iy1, ix2, iy2], [best_g["xmin"], best_g["ymin"], best_g["xmax"], best_g["ymax"]])
                        if primary_iou is None:
                            primary_iou = iou_val
                        new_boxes.append({
                            "class_id": best_g["class_id"],
                            "xmin": ix1,
                            "ymin": iy1,
                            "xmax": ix2,
                            "ymax": iy2,
                        })
                        alignment_rows.append({
                            "split": "train",
                            "file_name": base_name,
                            "match_type": match_type,
                            "class": ID_TO_CLASS[best_g["class_id"]],
                            "iou_isnet_vs_gt": round(iou_val, 5),
                            "gt_xmin": round(best_g["xmin"], 5),
                            "gt_ymin": round(best_g["ymin"], 5),
                            "gt_xmax": round(best_g["xmax"], 5),
                            "gt_ymax": round(best_g["ymax"], 5),
                            "isnet_xmin": round(ix1, 5),
                            "isnet_ymin": round(iy1, 5),
                            "isnet_xmax": round(ix2, 5),
                            "isnet_ymax": round(iy2, 5),
                            "isnet_area_ratio": round((ix2 - ix1) * (iy2 - iy1), 5),
                            "gt_area_ratio": round((best_g["xmax"] - best_g["xmin"]) * (best_g["ymax"] - best_g["ymin"]), 5),
                        })

                enc_write, fmt_write = apply_isnet_mask_to_bytes(encoded) if PIPELINE_MODE == "isnet_masked_rgb" else (encoded, fmt)
                tf_ex = make_tf_example(h, w, fname, src_id, enc_write, fmt_write, new_boxes)
                writer.write(tf_ex.SerializeToString())
                record_labeled_train_entry("train", base_name, w, h, new_boxes, f"isnet_{PIPELINE_MODE}", primary_iou)

                # Simpan 4 sampel preview lintas kelas
                used_cls = {m["class"] for m in preview_meta}
                cname0 = ID_TO_CLASS[new_boxes[0]["class_id"]]
                if len(preview_meta) < 4 and (cname0 not in used_cls or len(preview_meta) >= len(CLASSES)):
                    p_img = preview_dir / f"sample_{len(preview_meta)+1}.png"
                    Image.open(io.BytesIO(enc_write)).convert("RGB").save(p_img)
                    preview_meta.append({
                        "image_path": str(p_img),
                        "file_name": base_name,
                        "class": cname0,
                        "iou": round(float(primary_iou), 4) if primary_iou is not None else None,
                        "gt_box": [gt_boxes[0]["xmin"], gt_boxes[0]["ymin"], gt_boxes[0]["xmax"], gt_boxes[0]["ymax"]],
                        "isnet_box": [new_boxes[0]["xmin"], new_boxes[0]["ymin"], new_boxes[0]["xmax"], new_boxes[0]["ymax"]],
                    })

                for nb in new_boxes:
                    class_counts_train[ID_TO_CLASS[nb["class_id"]]] += 1
                stats["train_matched_isnet"] += 1
                stats["train_written_total"] += 1
                if PIPELINE_MODE == "isnet_masked_rgb" and stats["train_written_total"] % 100 == 0:
                    print(f"  Masking IS-Net progres: {stats['train_written_total']} gambar train...")
            else:
                if UNMATCHED_POLICY == "fallback_gt":
                    enc_write, fmt_write = apply_isnet_mask_to_bytes(encoded) if PIPELINE_MODE == "isnet_masked_rgb" else (encoded, fmt)
                    tf_ex = make_tf_example(h, w, fname, src_id, enc_write, fmt_write, gt_boxes)
                    writer.write(tf_ex.SerializeToString())
                    record_labeled_train_entry("train", base_name, w, h, gt_boxes, "fallback_gt", 1.0)
                    for g in gt_boxes:
                        class_counts_train[ID_TO_CLASS[g["class_id"]]] += 1
                    stats["train_unmatched_fallback_gt"] += 1
                    stats["train_written_total"] += 1
                else:
                    stats["train_unmatched_skipped"] += 1

    tmp_train.replace(out_train)

    if stats["train_matched_isnet"] == 0:
        raise RuntimeError(
            "0 gambar pada train.tfrecord yang cocok dengan output IS-Net! "
            "Periksa apakah nama file di 07_hasil_bounding_box_semua.json sama dengan data/raw/vehicles."
        )

    # Salin test.tfrecord GT bersih (tidak diubah) + audit alignment IS-Net pada split test
    shutil.copy2(v4_test, out_test)
    for raw in tf.data.TFRecordDataset(v4_test):
        stats["test_total"] += 1
        ex = tf.io.parse_single_example(raw, feature_spec)
        fname = ex["image/filename"].numpy().decode("utf-8", errors="replace")
        xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
        ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
        xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
        ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
        labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()
        if len(labels) == 0:
            stats["test_background"] += 1
            continue
        for cid in labels:
            class_counts_test[ID_TO_CLASS[int(cid)]] += 1
        isnet_rec, match_type = find_isnet_record(fname)
        if isnet_rec and isnet_rec["boxes"]:
            stats["test_matched_isnet_for_audit"] += 1
            g = [float(xmins[0]), float(ymins[0]), float(xmaxs[0]), float(ymaxs[0])]
            best_ib = max(isnet_rec["boxes"], key=lambda ib: compute_iou(ib["bbox_norm_xyxy"], g))
            ix1, iy1, ix2, iy2 = best_ib["bbox_norm_xyxy"]
            alignment_rows.append({
                "split": "test",
                "file_name": Path(fname.replace("\\", "/")).name,
                "match_type": match_type,
                "class": ID_TO_CLASS[int(labels[0])],
                "iou_isnet_vs_gt": round(compute_iou([ix1, iy1, ix2, iy2], g), 5),
                "gt_xmin": round(g[0], 5),
                "gt_ymin": round(g[1], 5),
                "gt_xmax": round(g[2], 5),
                "gt_ymax": round(g[3], 5),
                "isnet_xmin": round(ix1, 5),
                "isnet_ymin": round(iy1, 5),
                "isnet_xmax": round(ix2, 5),
                "isnet_ymax": round(iy2, 5),
                "isnet_area_ratio": round((ix2 - ix1) * (iy2 - iy1), 5),
                "gt_area_ratio": round((g[2] - g[0]) * (g[3] - g[1]), 5),
            })

else:
    mode_used = f"standalone_labeled_{PIPELINE_MODE}"
    print(f"TFRecord v4 tidak ditemukan. Beralih ke mode mandiri: {mode_used}")
    img_root = Path(params["images_dir"])
    if not img_root.is_dir():
        raise FileNotFoundError(
            f"Folder gambar tidak ditemukan ({img_root}) dan TFRecord v4 juga tidak tersedia. "
            "Upload folder MyDrive/ferqueue/pcv-coco-colab-v4 atau siapkan gambar + kolom Label pada CSV IS-Net."
        )
    img_index = {}
    for p in img_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png"):
            img_index[p.name] = p
            img_index[p.name.lower()] = p

    labeled_items = []
    for rec in isnet_records:
        fn = rec["file_name"]
        img_p = img_index.get(fn) or img_index.get(fn.lower())
        if not img_p:
            continue
        valid_objs = [b for b in rec["boxes"] if b.get("label") in CLASS_TO_ID]
        if not valid_objs:
            continue
        labeled_items.append({"img_path": img_p, "file_name": fn, "boxes": valid_objs, "primary_class": valid_objs[0]["label"]})

    if len(labeled_items) < 10:
        raise RuntimeError(
            "Mode mandiri membutuhkan minimal 10 gambar berlabel kelas (Mobil/Truk/Bus) pada CSV/JSON IS-Net. "
            "Untuk cara paling mudah, pastikan folder MyDrive/ferqueue/pcv-coco-colab-v4 sudah tersedia di Google Drive."
        )

    rng = random.Random(SEED)
    by_cls = defaultdict(list)
    for it in labeled_items:
        by_cls[it["primary_class"]].append(it)
    train_items, test_items = [], []
    for cls_name, bucket in sorted(by_cls.items()):
        rng.shuffle(bucket)
        n_test = max(1, int(round(len(bucket) * 0.3)))
        test_items.extend(bucket[:n_test])
        train_items.extend(bucket[n_test:])
    rng.shuffle(train_items)
    rng.shuffle(test_items)

    def write_standalone_split(items, dst_path, counter, split_name, collect_preview=False):
        tmp = dst_path.with_suffix(".tfrecord.partial")
        with tf.io.TFRecordWriter(str(tmp)) as writer:
            for it in items:
                with Image.open(it["img_path"]) as im:
                    rgb = ImageOps.exif_transpose(im).convert("RGB")
                    w, h = rgb.size
                    buf = io.BytesIO()
                    rgb.save(buf, format="PNG")
                    encoded = buf.getvalue()
                enc_write, fmt_write = (
                    apply_isnet_mask_to_bytes(encoded)
                    if (PIPELINE_MODE == "isnet_masked_rgb" and split_name == "train")
                    else (encoded, b"png")
                )
                boxes = [
                    {
                        "class_id": CLASS_TO_ID[b["label"]],
                        "xmin": b["bbox_norm_xyxy"][0],
                        "ymin": b["bbox_norm_xyxy"][1],
                        "xmax": b["bbox_norm_xyxy"][2],
                        "ymax": b["bbox_norm_xyxy"][3],
                    }
                    for b in it["boxes"]
                ]
                src_id = hashlib.sha256(enc_write).hexdigest()
                tf_ex = make_tf_example(h, w, it["file_name"], src_id, enc_write, fmt_write, boxes)
                writer.write(tf_ex.SerializeToString())
                if split_name == "train":
                    record_labeled_train_entry("train", it["file_name"], w, h, boxes, f"isnet_{PIPELINE_MODE}", None)
                for b in boxes:
                    counter[ID_TO_CLASS[b["class_id"]]] += 1
                if collect_preview and len(preview_meta) < 4:
                    p_img = preview_dir / f"sample_{len(preview_meta)+1}.png"
                    Image.open(io.BytesIO(enc_write)).convert("RGB").save(p_img)
                    preview_meta.append({
                        "image_path": str(p_img),
                        "file_name": it["file_name"],
                        "class": it["primary_class"],
                        "iou": None,
                        "gt_box": None,
                        "isnet_box": it["boxes"][0]["bbox_norm_xyxy"],
                    })
        tmp.replace(dst_path)

    write_standalone_split(train_items, out_train, class_counts_train, "train", collect_preview=True)
    write_standalone_split(test_items, out_test, class_counts_test, "test", collect_preview=False)
    stats = {
        "train_source_total": len(labeled_items),
        "train_matched_isnet": len(train_items),
        "train_background_kept": 0,
        "train_unmatched_skipped": 0,
        "train_unmatched_fallback_gt": 0,
        "train_written_total": len(train_items),
        "test_total": len(test_items),
        "test_background": 0,
        "test_matched_isnet_for_audit": len(test_items),
    }

train_ious = [r["iou_isnet_vs_gt"] for r in alignment_rows if r["split"] == "train"]
test_ious = [r["iou_isnet_vs_gt"] for r in alignment_rows if r["split"] == "test"]
iou_summary = {}
for name, arr in (("train", train_ious), ("test", test_ious)):
    if arr:
        a = np.asarray(arr, dtype=np.float64)
        iou_summary[name] = {
            "count": int(a.size),
            "mean_iou": round(float(np.mean(a)), 4),
            "median_iou": round(float(np.median(a)), 4),
            "pct_iou_ge_50": round(float(100.0 * np.mean(a >= 0.50)), 2),
            "pct_iou_ge_75": round(float(100.0 * np.mean(a >= 0.75)), 2),
            "pct_iou_lt_50": round(float(100.0 * np.mean(a < 0.50)), 2),
        }

audit_payload = {
    "mode": mode_used,
    "training_pipeline_mode": PIPELINE_MODE,
    "isnet_source": isnet_payload["source"],
    "unmatched_policy": UNMATCHED_POLICY,
    "keep_background_negative": KEEP_BG,
    "stats": stats,
    "train_class_counts": dict(class_counts_train),
    "test_class_counts": dict(class_counts_test),
    "isnet_vs_gt_iou_summary": iou_summary,
    "preview_samples": preview_meta,
}

# Simpan audit + dataset berlabel (CSV, JSON, COCO JSON)
for p_out in (params["audit_json"], params["labeled_csv"], params["labeled_json"], params["labeled_coco_json"]):
    Path(p_out).parent.mkdir(parents=True, exist_ok=True)

Path(params["audit_json"]).write_text(json.dumps(audit_payload, indent=2, ensure_ascii=False), encoding="utf-8")

if alignment_rows:
    with open(params["audit_csv"], "w", newline="", encoding="utf-8") as fh:
        w_csv = csv.DictWriter(fh, fieldnames=list(alignment_rows[0].keys()))
        w_csv.writeheader()
        w_csv.writerows(alignment_rows)

if labeled_csv_rows:
    with open(params["labeled_csv"], "w", newline="", encoding="utf-8") as fh:
        w_csv = csv.DictWriter(fh, fieldnames=list(labeled_csv_rows[0].keys()))
        w_csv.writeheader()
        w_csv.writerows(labeled_csv_rows)

Path(params["labeled_json"]).write_text(
    json.dumps(
        {
            "pipeline": f"pcv_isnet_berlabel_{PIPELINE_MODE}",
            "classes": CLASSES,
            "total_images": len(labeled_json_results),
            "class_counts": dict(class_counts_train),
            "results": labeled_json_results,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
Path(params["labeled_coco_json"]).write_text(
    json.dumps(coco_train_doc, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n=== RINGKASAN DATASET BERLABEL & TFRECORD IS-NET ===")
print(json.dumps({k: v for k, v in audit_payload.items() if k != "preview_samples"}, indent=2, ensure_ascii=False))
print("\nDataset BBox IS-Net Berlabel tersimpan di:")
print("  CSV       :", params["labeled_csv"])
print("  JSON      :", params["labeled_json"])
print("  COCO JSON :", params["labeled_coco_json"])
""",
    encoding="utf-8",
)

run_venv([str(builder_script), str(builder_params_path)])

audit_data = json.loads(AUDIT_JSON_PATH.read_text(encoding="utf-8"))
TRAIN_COUNT = int(audit_data["stats"]["train_written_total"])
TEST_COUNT = int(audit_data["stats"]["test_total"])
STEPS_PER_EPOCH = max(1, math.ceil(TRAIN_COUNT / BATCH_SIZE))
NUM_STEPS = EPOCHS * STEPS_PER_EPOCH

training_plan = {
    "training_pipeline_mode": TRAINING_PIPELINE_MODE,
    "train_images": TRAIN_COUNT,
    "test_images": TEST_COUNT,
    "batch_size": BATCH_SIZE,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "num_steps": NUM_STEPS,
    "requested_epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "freeze_batchnorm": FREEZE_BATCHNORM,
    "source_mode": audit_data["mode"],
    "unmatched_policy": UNMATCHED_POLICY,
}
(DRIVE_RUN_DIR / "training-plan.json").write_text(json.dumps(training_plan, indent=2), encoding="utf-8")
shutil.copy2(LOCAL_LABEL_MAP, DRIVE_RUN_DIR / "label_map.pbtxt")

print(f"\nRencana training siap: {TRAIN_COUNT} gambar train, {TEST_COUNT} gambar test -> {NUM_STEPS} steps ({EPOCHS} epoch x {STEPS_PER_EPOCH} steps/epoch)")

# Tampilkan preview tabel dataset BBox IS-Net yang sudah berlabel
df_labeled = pd.read_csv(LABELED_CSV_PATH)
print("\nContoh 5 baris dataset BBox IS-Net yang sudah berlabel:")
display(df_labeled.head(5)[["No", "File_Name", "Label", "BBox_xmin", "BBox_ymin", "BBox_xmax", "BBox_ymax", "IoU_ISNet_vs_GT"]])

# Visualisasi 4 sampel data latih dengan TAG LABEL KELAS di atas BBox IS-Net
CLASS_RGB = {"Mobil": (59, 130, 246), "Truk": (245, 158, 11), "Bus": (16, 185, 129)}
samples = audit_data.get("preview_samples", [])
if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(4.7 * len(samples), 4.9))
    if len(samples) == 1:
        axes = [axes]
    for ax, s in zip(axes, samples):
        im = np.asarray(Image.open(s["image_path"])).copy()
        h, w = im.shape[:2]
        ix1, iy1, ix2, iy2 = [int(round(v)) for v in (
            s["isnet_box"][0] * w, s["isnet_box"][1] * h, s["isnet_box"][2] * w, s["isnet_box"][3] * h
        )]
        col = CLASS_RGB.get(s["class"], (34, 197, 94))
        thick = max(2, int(round(min(h, w) / 160)))
        cv2.rectangle(im, (ix1, iy1), (ix2, iy2), col, thick)
        tag = f"{s['class']} (IS-Net)"
        fscale = max(0.55, min(h, w) / 680.0)
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, fscale, max(1, thick - 1))
        ly = max(th + 8, iy1)
        cv2.rectangle(im, (ix1, ly - th - 8), (ix1 + tw + 10, ly), col, -1)
        cv2.putText(im, tag, (ix1 + 5, ly - 4), cv2.FONT_HERSHEY_SIMPLEX, fscale, (255, 255, 255), max(1, thick - 1), cv2.LINE_AA)
        ax.imshow(im)
        if s.get("gt_box"):
            gx1, gy1, gx2, gy2 = s["gt_box"]
            ax.add_patch(plt.Rectangle(
                (gx1 * w, gy1 * h), (gx2 - gx1) * w, (gy2 - gy1) * h,
                fill=False, edgecolor="#FACC15", linewidth=2.0, linestyle="--", label="Ref GT Manual",
            ))
        iou_txt = f"IoU={s['iou']:.2f}" if s.get("iou") is not None else "Standalone"
        ax.set_title(f"Label: {s['class']} | {iou_txt}\n{s['file_name'][:24]}", fontsize=9.5, fontweight="bold")
        ax.axis("off")

    fig.suptitle(
        f"Verifikasi Visual Data Latih Berlabel ({TRAINING_PIPELINE_MODE}): Kotak Solid = BBox IS-Net Berlabel | Kuning = Ref GT",
        fontsize=11.8, fontweight="bold",
    )
    plt.tight_layout(rect=[0, 0.02, 1, 0.93])
    fig.savefig(LABELED_EXPORT_DIR / "01_verifikasi_bbox_isnet_berlabel.png", dpi=220, bbox_inches="tight", facecolor="white")
    plt.show()

## Cell 6 — Unduh Pretrained SSD MobileNet V2 FPNLite & Buat `pipeline.config`

In [ ]:
import shutil
import tarfile
import urllib.request
from pathlib import Path

PRETRAINED_DIR = WORK_DIR / "pretrained"
PRETRAINED_MODEL_DIR = PRETRAINED_DIR / MODEL_NAME
PRETRAINED_URL = f"https://storage.googleapis.com/download.tensorflow.org/models/object_detection/tf2/20200711/{MODEL_NAME}.tar.gz"
PIPELINE_CONFIG_PATH = WORK_DIR / "pipeline.config"

if not (PRETRAINED_MODEL_DIR / "checkpoint/ckpt-0.index").is_file():
    PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
    archive = PRETRAINED_DIR / f"{MODEL_NAME}.tar.gz"
    print(f"Mengunduh pretrained {MODEL_NAME}...")
    urllib.request.urlretrieve(PRETRAINED_URL, archive)
    with tarfile.open(archive, "r:gz") as tf_tar:
        tf_tar.extractall(PRETRAINED_DIR)
    archive.unlink(missing_ok=True)
    print("Pretrained siap:", PRETRAINED_MODEL_DIR)
else:
    print("Pretrained sudah tersedia:", PRETRAINED_MODEL_DIR)

make_cfg_script = WORK_DIR / "make_pipeline.py"
make_cfg_script.write_text(
    f"""from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2, train_pb2

pretrained = Path({str(PRETRAINED_MODEL_DIR)!r})
pipeline = pipeline_pb2.TrainEvalPipelineConfig()
text_format.Parse((pretrained / "pipeline.config").read_text(encoding="utf-8"), pipeline)

pipeline.model.ssd.num_classes = {len(CLASSES)}
pipeline.model.ssd.freeze_batchnorm = {FREEZE_BATCHNORM!r}

train = pipeline.train_config
train.batch_size = {BATCH_SIZE}
train.num_steps = {NUM_STEPS}
train.fine_tune_checkpoint = str(pretrained / "checkpoint/ckpt-0")
train.fine_tune_checkpoint_type = "detection"
train.fine_tune_checkpoint_version = train_pb2.CheckpointVersion.V2
train.sync_replicas = False
train.replicas_to_aggregate = 1

train.optimizer.Clear()
momentum = train.optimizer.momentum_optimizer
momentum.momentum_optimizer_value = 0.9
schedule = momentum.learning_rate.cosine_decay_learning_rate
schedule.learning_rate_base = {LEARNING_RATE}
schedule.total_steps = {NUM_STEPS}
schedule.warmup_steps = {min(100, max(1, NUM_STEPS // 10))}
schedule.warmup_learning_rate = {LEARNING_RATE / 10}
train.optimizer.use_moving_average = False

del train.data_augmentation_options[:]
if {HORIZONTAL_FLIP!r}:
    train.data_augmentation_options.add().random_horizontal_flip.SetInParent()
if {BRIGHTNESS_DELTA!r}:
    train.data_augmentation_options.add().random_adjust_brightness.max_delta = {BRIGHTNESS_DELTA}

train.max_number_of_boxes = 100

label_map = {str(LOCAL_LABEL_MAP)!r}
pipeline.train_input_reader.label_map_path = label_map
del pipeline.train_input_reader.tf_record_input_reader.input_path[:]
pipeline.train_input_reader.tf_record_input_reader.input_path.append({str(LOCAL_TRAIN_TFREC)!r})
pipeline.train_input_reader.num_readers = 1

for reader in pipeline.eval_input_reader:
    reader.label_map_path = label_map
    reader.shuffle = False
    reader.num_epochs = 1
    reader.num_readers = 1
    del reader.tf_record_input_reader.input_path[:]
    reader.tf_record_input_reader.input_path.append({str(LOCAL_TEST_TFREC)!r})

pipeline.eval_config.use_moving_averages = False
pipeline.eval_config.batch_size = 1

out = Path({str(PIPELINE_CONFIG_PATH)!r})
out.write_text(text_format.MessageToString(pipeline), encoding="utf-8")
print("pipeline.config dibuat di:", out)
""",
    encoding="utf-8",
)

run_venv([str(make_cfg_script)])
shutil.copy2(PIPELINE_CONFIG_PATH, DRIVE_RUN_DIR / "pipeline.config")
print("Salinan pipeline.config disimpan juga ke:", DRIVE_RUN_DIR / "pipeline.config")

## Cell 7 — (Opsional) Jalankan TensorBoard untuk Pantau Kurva Loss

Jalankan cell ini sebelum Cell 8 jika ingin memantau `Loss/classification_loss`, `Loss/localization_loss`, dan `Loss/total_loss` secara langsung. Boleh dilewati jika tidak perlu.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{MODEL_DIR}"

## Cell 8 — Training SSD MobileNet V2 FPNLite di GPU (dari Dataset BBox Berlabel IS-Net)

- Di GPU T4 Colab, 30 epoch (~4.080 steps) umumnya memakan waktu **10–20 menit**.
- Checkpoint disimpan tiap **100 steps** ke Google Drive (`DRIVE_RUN_DIR/training`). Jika Colab terputus, jalankan ulang Cell 1–6 lalu Cell 8 untuk melanjutkan (*resume*) otomatis dari checkpoint terakhir.

In [ ]:
print(f"Memulai training SSD MobileNet V2 ({TRAINING_PIPELINE_MODE}) hingga {NUM_STEPS} langkah...")
print(f"Checkpoint disimpan di: {MODEL_DIR}")

run_venv([
    "-m", "object_detection.model_main_tf2",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--model_dir={MODEL_DIR}",
    f"--num_train_steps={NUM_STEPS}",
    f"--checkpoint_every_n={CHECKPOINT_EVERY_N}",
    "--alsologtostderr",
])

print("\nTraining selesai!")

## Cell 9 — Export `SavedModel` & Evaluasi COCO pada `test.tfrecord`

Mengekspor checkpoint terakhir menjadi TensorFlow `SavedModel` (`--input_type=image_tensor`), lalu menghitung metrik COCO resmi (`mAP 50:95`, `AP@0.50`, `AP@0.75`, `AR@100`, dan mAP per kelas `Mobil`, `Truk`, `Bus`) identik dengan `train_colab_v4.ipynb` dan `pcv.py evaluate`.

In [ ]:
import json
from pathlib import Path

find_ckpt = run_venv(
    [
        "-c",
        f"import tensorflow as tf; print(tf.train.latest_checkpoint({str(MODEL_DIR)!r}) or '')",
    ],
    stream=False,
)
latest_ckpt = find_ckpt.stdout.strip().splitlines()[-1].strip()
if not latest_ckpt:
    raise RuntimeError(f"Tidak ditemukan checkpoint di {MODEL_DIR}. Pastikan Cell 8 sudah selesai.")

CKPT_NAME = Path(latest_ckpt).name
EXPORT_DIR = WORK_DIR / "exports" / CKPT_NAME
SAVED_MODEL_DIR = EXPORT_DIR / "saved_model"

print(f"Checkpoint terakhir: {CKPT_NAME}")
print(f"Mengekspor SavedModel ke: {EXPORT_DIR} ...")

run_venv([
    "-m", "object_detection.exporter_main_v2",
    "--input_type=image_tensor",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--trained_checkpoint_dir={MODEL_DIR}",
    f"--output_directory={EXPORT_DIR}",
])

if not (SAVED_MODEL_DIR / "saved_model.pb").is_file():
    raise RuntimeError("Export gagal: saved_model.pb tidak terbentuk.")
print(f"Export berhasil! SavedModel siap di: {SAVED_MODEL_DIR}\n")

REPORT_PATH = REPORTS_DIR / f"{CKPT_NAME}-test.json"
eval_params_path = WORK_DIR / "eval_params.json"
eval_params_path.write_text(
    json.dumps(
        {
            "classes": CLASSES,
            "training_pipeline_mode": TRAINING_PIPELINE_MODE,
            "saved_model_dir": str(SAVED_MODEL_DIR),
            "test_tfrecord": str(LOCAL_TEST_TFREC),
            "report_path": str(REPORT_PATH),
            "latest_report_path": str(REPORTS_DIR / "latest-test.json"),
            "checkpoint": CKPT_NAME,
        },
        indent=2,
    ),
    encoding="utf-8",
)

eval_script = WORK_DIR / "evaluate_test.py"
eval_script.write_text(
    r"""import copy
import json
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

params = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CLASSES = params["classes"]
SAVED_MODEL_DIR = Path(params["saved_model_dir"])
TEST_TFREC = Path(params["test_tfrecord"])
REPORT_PATH = Path(params["report_path"])
LATEST_REPORT_PATH = Path(params["latest_report_path"])
CKPT_NAME = params["checkpoint"]

SUMMARY_NAMES = (
    "Precision/mAP", "Precision/mAP@.50IOU", "Precision/mAP@.75IOU",
    "Precision/mAP (small)", "Precision/mAP (medium)", "Precision/mAP (large)",
    "Recall/AR@1", "Recall/AR@10", "Recall/AR@100",
    "Recall/AR@100 (small)", "Recall/AR@100 (medium)", "Recall/AR@100 (large)",
)

feature_spec = {
    "image/height": tf.io.FixedLenFeature([], tf.int64),
    "image/width": tf.io.FixedLenFeature([], tf.int64),
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

print("Memuat SavedModel:", SAVED_MODEL_DIR)
model = tf.saved_model.load(str(SAVED_MODEL_DIR))
detector = model.signatures["serving_default"]

categories = [{"id": idx, "name": name} for idx, name in enumerate(CLASSES, 1)]
groundtruth = {"info": {}, "images": [], "annotations": [], "categories": categories}
predictions = []

dataset = tf.data.TFRecordDataset(str(TEST_TFREC))
image_count = 0
for image_id, raw_record in enumerate(dataset, 1):
    ex = tf.io.parse_single_example(raw_record, feature_spec)
    height = int(ex["image/height"].numpy())
    width = int(ex["image/width"].numpy())
    filename = ex["image/filename"].numpy().decode("utf-8", errors="replace")
    img = tf.io.decode_image(ex["image/encoded"], channels=3, expand_animations=False)
    img_tensor = tf.cast(img, tf.uint8)[None, ...]

    result = detector(img_tensor)
    count = int(result["num_detections"][0])
    det_boxes = result["detection_boxes"][0, :count].numpy()
    det_classes = result["detection_classes"][0, :count].numpy()
    det_scores = result["detection_scores"][0, :count].numpy()

    groundtruth["images"].append({"id": image_id, "file_name": filename, "width": width, "height": height})

    xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
    xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
    ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
    ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
    labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()

    for xmin, ymin, xmax, ymax, cid in zip(xmins, ymins, xmaxs, ymaxs, labels):
        x1, y1, x2, y2 = float(xmin * width), float(ymin * height), float(xmax * width), float(ymax * height)
        bw, bh = x2 - x1, y2 - y1
        groundtruth["annotations"].append({
            "id": len(groundtruth["annotations"]) + 1,
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [x1, y1, bw, bh],
            "area": bw * bh,
            "iscrowd": 0,
        })

    scale = np.array([height, width, height, width], dtype=np.float32)
    for coords, cid, score in zip(det_boxes * scale, det_classes, det_scores):
        ymin, xmin, ymax, xmax = map(float, coords)
        predictions.append({
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
            "score": float(score),
        })
    image_count = image_id
    if image_id % 100 == 0:
        print(f"Dievaluasi {image_id} gambar test...")

reference = COCO()
reference.dataset = copy.deepcopy(groundtruth)
reference.createIndex()
if predictions:
    predicted = reference.loadRes(copy.deepcopy(predictions))
else:
    predicted = COCO()
    predicted.dataset = {
        "images": copy.deepcopy(reference.dataset["images"]),
        "categories": copy.deepcopy(reference.dataset["categories"]),
        "annotations": [],
    }
    predicted.createIndex()

evaluator = COCOeval(reference, predicted, iouType="bbox")
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

metrics = {"DetectionBoxes_" + name: float(val) for name, val in zip(SUMMARY_NAMES, evaluator.stats)}
precision = evaluator.eval["precision"]
area_idx = evaluator.params.areaRngLbl.index("all")
max_idx = list(evaluator.params.maxDets).index(100)
names = {cat["id"]: cat["name"] for cat in reference.dataset["categories"]}
for idx, cid in enumerate(evaluator.params.catIds):
    vals = precision[:, :, idx, area_idx, max_idx]
    valid = vals[vals > -1]
    metrics["DetectionBoxes_PerformanceByCategory/mAP/" + names[cid]] = float(np.mean(valid)) if valid.size else -1.0

report = {
    "split": "test",
    "training_pipeline_mode": params["training_pipeline_mode"],
    "checkpoint": CKPT_NAME,
    "image_count": image_count,
    "metrics": metrics,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
LATEST_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== RINGKASAN METRIK TEST (MODEL SSD IS-NET) ===")
print(f"Mode       : {params['training_pipeline_mode']}")
print(f"Checkpoint : {CKPT_NAME} ({image_count} gambar)")
print(f"mAP 50:95  : {metrics['DetectionBoxes_Precision/mAP']:.4f}")
print(f"AP@0.50    : {metrics['DetectionBoxes_Precision/mAP@.50IOU']:.4f}")
print(f"AP@0.75    : {metrics['DetectionBoxes_Precision/mAP@.75IOU']:.4f}")
for cls_name in CLASSES:
    key = f"DetectionBoxes_PerformanceByCategory/mAP/{cls_name}"
    print(f"mAP {cls_name:<6} : {metrics[key]:.4f}")
print("\nLaporan tersimpan di:", REPORT_PATH)
""",
    encoding="utf-8",
)

run_venv([str(eval_script), str(eval_params_path)])

## Cell 10 — Paketkan `pcv-model-isnet-<ckpt>.zip` & `receiver-isnet.json` Siap Dipasang ke Proyek FERQUEUE

Paket ZIP ini divalidasi langsung terhadap seluruh aturan ketat `apps/computer-vision/src/ferqueue_cv/sharing.py` (`model.json` + `saved_model/**`), sehingga bisa langsung diimpor ke laptop proyek FERQUEUE dengan perintah `pcv.py import-model` tanpa error, serta langsung bisa dibandingkan di **Notebook 8 (`8_perbandingan_ssd_isnet_vs_sebelumnya.ipynb`)**.

In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path, PurePosixPath

PREFIX_ZIP = "pcv-model-isnet" if TRAINING_PIPELINE_MODE == "isnet_bbox_rgb" else "pcv-model-isnet-masked"
ARCHIVE_NAME = f"{PREFIX_ZIP}-{CKPT_NAME}.zip"
LOCAL_ARCHIVE = WORK_DIR / ARCHIVE_NAME
DRIVE_ARCHIVE = DRIVE_RUN_DIR / ARCHIVE_NAME

public_meta = {
    "classes": CLASSES,
    "model": MODEL_NAME,
    "checkpoint": CKPT_NAME,
    "preprocessing": PREPROCESSING,
}

# 1. Buat arsip ZIP sesuai format persis ferqueue_cv/sharing.py
with zipfile.ZipFile(LOCAL_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("model.json", json.dumps(public_meta, ensure_ascii=False, indent=2))
    for path in sorted(SAVED_MODEL_DIR.rglob("*")):
        if path.is_file():
            arcname = "saved_model/" + path.relative_to(SAVED_MODEL_DIR).as_posix()
            zf.write(path, arcname)

# 2. Validasi mandiri identik dengan ferqueue_cv/sharing.py agar 100% diterima pcv.py import-model
with zipfile.ZipFile(LOCAL_ARCHIVE) as zf:
    names = zf.namelist()
    assert len(names) == len(set(names)), "Terdapat nama file duplikat di arsip."
    for n in names:
        p = PurePosixPath(n)
        assert "\\" not in n and not p.is_absolute() and ".." not in p.parts and ":" not in n
        assert n == "model.json" or (p.parts and p.parts[0] == "saved_model")
    assert "model.json" in names
    assert "saved_model/saved_model.pb" in names
    assert any(n.startswith("saved_model/variables/") for n in names)
    meta = json.loads(zf.read("model.json").decode("utf-8"))
    assert isinstance(meta["classes"], list) and len(meta["classes"]) > 0
    assert re.fullmatch(r"ckpt-\d+", meta["checkpoint"])
    assert meta["model"] == "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"

# 3. Salin arsip ke Google Drive
shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

# 4. Buat file konfigurasi receiver-isnet.json siap pakai untuk proyek lokal FERQUEUE
run_name_local = "pcv-receiver-isnet" if TRAINING_PIPELINE_MODE == "isnet_bbox_rgb" else "pcv-receiver-isnet-masked"
receiver_config = {
    "dataset_format": "coco",
    "dataset_dir": "data/raw/vehicles",
    "coco_annotations": {
        "train": "data/processed/pcv-coco-v4/annotations/instances_train.json",
        "test": "data/processed/pcv-coco-v4/annotations/instances_test.json",
    },
    "prepared_dir": "data/processed/pcv-coco-manifest-v4",
    "run_dir": f"models/mobilenet/runs/{run_name_local}",
    "seed": 42,
    "train_ratio": 0.7,
    "test_ratio": 0.3,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "freeze_batchnorm": FREEZE_BATCHNORM,
    "checkpoint_every_n": CHECKPOINT_EVERY_N,
    "augmentation": {
        "horizontal_flip": HORIZONTAL_FLIP,
        "brightness_delta": BRIGHTNESS_DELTA,
    },
    "preprocessing": PREPROCESSING,
    "segmentation": {
        "iterations": 5,
        "morphology_kernel": 3,
    },
    "confidence_threshold": 0.5,
    "nms_iou_threshold": 0.45,
    "max_detections": 10,
    "class_agnostic_nms": True,
}
RECEIVER_CFG_DRIVE = DRIVE_RUN_DIR / "receiver-isnet.json"
RECEIVER_CFG_DRIVE.write_text(json.dumps(receiver_config, indent=2), encoding="utf-8")

size_mb = DRIVE_ARCHIVE.stat().st_size / (1024**2)
print("=" * 74)
print("PAKET MODEL SSD MOBILENET V2 (IS-NET) VALID & SIAP DIPASANG KE FERQUEUE!")
print("=" * 74)
print(f"Mode Pipeline      : {TRAINING_PIPELINE_MODE}")
print(f"Arsip ZIP di Drive : {DRIVE_ARCHIVE} ({size_mb:.1f} MB)")
print(f"Config receiver    : {RECEIVER_CFG_DRIVE}")
print(f"Dataset Berlabel   : {LABELED_EXPORT_DIR}")
print("\n=== CARA MEMASANG KE PROYEK LOKAL FERQUEUE ===")
print(f"1. Unduh {ARCHIVE_NAME} ke folder: outputs/{ARCHIVE_NAME}")
print("2. Unduh receiver-isnet.json ke: apps/computer-vision/configs/receiver-isnet.json")
print("3. Jalankan impor model di terminal root proyek FERQUEUE:")
print(f"   .venv-pcv/bin/python pcv.py import-model --config apps/computer-vision/configs/receiver-isnet.json --archive outputs/{ARCHIVE_NAME}")
print("4. Uji prediksi / aktifkan di proyek:")
print("   - Uji gambar : .venv-pcv/bin/python pcv.py predict --config apps/computer-vision/configs/receiver-isnet.json --input data/helper --output outputs/isnet-predictions")
print("   - Live API   : ubah CV_CONFIG_PATH=apps/computer-vision/configs/receiver-isnet.json di file .env")
print("\n=== LANJUT KE PERBANDINGAN PROYEK (NOTEBOOK 8) ===")
print("Buka apps/computer-vision/notebooks/8_perbandingan_ssd_isnet_vs_sebelumnya.ipynb")
print("Notebook 8 akan membandingkan: Proyek Saat Ini (v4) vs Proyek + SSD IS-Net (Training) vs Proyek + IS-Net Aktif di Runtime!")

# Opsional: hapus tanda '#' di bawah jika ingin langsung unduh ZIP lewat browser Colab
# from google.colab import files
# files.download(str(LOCAL_ARCHIVE))